<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2009%20-%202026-05-05%20-%20Regresiones/clase_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 9 · Regresión lineal, polinomial y multicolinealidad (VIF)

**Seminario EDA 2026 — Semana 3, Día 2 (martes 5 de mayo)**

> En esta clase pasamos del *describir* al *predecir*. Construimos el primer modelo predictivo del seminario — la regresión lineal — y aprendemos:
>
> 1. Cómo se ajusta una recta a los datos (intuición y mecánica del OLS).  
> 2. Cómo extender el modelo a múltiples predictores y a relaciones no lineales (polinomios).  
> 3. Cómo detectar — y tratar — la **multicolinealidad** con el factor **VIF**.
>
> **Entrega parcial S3 (avance):** una regresión lineal y una múltiple sobre el dataset del proyecto, con análisis VIF.

Acompaña esta clase con los simuladores:  
🎮 [Lineal simple](../simuladores/simulador-regresion-lineal.html) · [Múltiple 3D](../simuladores/simulador-regresion-3D-advertising.html) · [VIF](../simuladores/simulador-vif.html) · [Overfitting](../simuladores/simulador-overfitting.html)

## 0. Setup

Si estás en Colab y `statsmodels` no está, descomenta la primera línea.

In [ ]:
# !pip install -q statsmodels scikit-learn matplotlib seaborn pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import r2_score, mean_squared_error

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

SEED = 42
np.random.seed(SEED)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
print('Setup OK')

## 1. Dataset Advertising (TV, Radio, Newspaper → Sales)

Es el dataset clásico del libro *An Introduction to Statistical Learning* (Hastie, Tibshirani, Witten, James). 200 observaciones de inversión publicitaria semanal de una empresa, en miles de USD por canal, y las ventas resultantes en miles de unidades.

In [ ]:
URL = 'https://raw.githubusercontent.com/justmarkham/scikit-learn-videos/master/data/Advertising.csv'
df = pd.read_csv(URL, index_col=0)
print(df.shape)
df.head()

In [ ]:
# Descripción rápida y matriz de correlación
print(df.describe().round(2))
print('\nMatriz de correlación:')
df.corr().round(3)

In [ ]:
# Pairplot — hacemos primero la inspección visual antes de modelar
sns.pairplot(df, diag_kind='kde', plot_kws={'alpha': 0.6})
plt.suptitle('Advertising — pairplot', y=1.02)
plt.show()

**Lectura del pairplot:**
- TV vs Sales — relación clara, casi lineal pero con ligera curvatura (saturación en valores altos).
- Radio vs Sales — relación positiva pero más dispersa.
- Newspaper vs Sales — la nube es bastante difusa; correlación ~0.23.

En el resto del notebook iremos viendo cómo los modelos lineales capturan (o no) cada una de estas relaciones.

## 2. Regresión lineal simple — TV → Sales

Modelo: $\hat{y} = \beta_0 + \beta_1 \cdot \text{TV}$  
OLS minimiza $\sum (y_i - \hat{y}_i)^2$.

In [ ]:
X = df[['TV']].values
y = df['Sales'].values

modelo_simple = LinearRegression().fit(X, y)
print(f'β0 (intercept) = {modelo_simple.intercept_:.4f}')
print(f'β1 (TV)         = {modelo_simple.coef_[0]:.4f}')
print(f'R²              = {modelo_simple.score(X, y):.4f}')

In [ ]:
# Visualización: scatter + recta de OLS
xs = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
ys = modelo_simple.predict(xs)

plt.scatter(X, y, alpha=0.6, label='Datos reales')
plt.plot(xs, ys, color='crimson', linewidth=2,
         label=f'OLS:  y = {modelo_simple.intercept_:.2f} + {modelo_simple.coef_[0]:.4f}·TV')
plt.xlabel('Inversión en TV (miles USD)')
plt.ylabel('Sales (miles unidades)')
plt.title('Regresión lineal simple — Advertising')
plt.legend(); plt.tight_layout(); plt.show()

**Interpretación de negocio:**  
Cada $1,000 USD adicionales invertidos en TV se asocian con un aumento de aprox. 47.5 unidades vendidas. Con cero inversión en TV se esperan ~7,030 unidades (línea base — clientes que llegan por otros medios).

### 2.1 Reto: replica el modelo con Radio

Cambia `'TV'` por `'Radio'` y compara intercept, pendiente y R².

In [ ]:
# TODO: reemplaza 'TV' por 'Radio' y reporta los coeficientes
X_radio = df[['___']].values   # ← completa
modelo_radio = LinearRegression().fit(X_radio, y)
print(f'β0 = {modelo_radio.intercept_:.4f}')
print(f'β1 = {modelo_radio.coef_[0]:.4f}')
print(f'R² = {modelo_radio.score(X_radio, y):.4f}')

In [ ]:
# Asserts de aceptación (cuando completes la celda anterior se cumplen)
# assert modelo_radio.coef_[0] > 0, 'Esperabas pendiente positiva'
# assert 0.2 < modelo_radio.score(X_radio, y) < 0.5, 'R² fuera de rango razonable'
# print('✓ Ejercicio completado')

## 3. Regresión lineal múltiple — TV + Radio + Newspaper → Sales

Modelo: $\hat{y} = \beta_0 + \beta_1 \text{TV} + \beta_2 \text{Radio} + \beta_3 \text{Newspaper}$

In [ ]:
X = df[['TV', 'Radio', 'Newspaper']].values
y = df['Sales'].values

modelo_mult = LinearRegression().fit(X, y)
print('Coeficientes:')
for nombre, beta in zip(['TV', 'Radio', 'Newspaper'], modelo_mult.coef_):
    print(f'  β_{nombre:<10s} = {beta:+.4f}')
print(f'\nIntercept β0 = {modelo_mult.intercept_:.4f}')
print(f'R²            = {modelo_mult.score(X, y):.4f}')

### 3.1 Reporte completo con statsmodels

`statsmodels.OLS` da p-values, intervalos de confianza, R² ajustado, F-test global y mucho más. Es indispensable cuando interpretamos coeficientes.

In [ ]:
X_sm = add_constant(df[['TV', 'Radio', 'Newspaper']])
ols = sm.OLS(y, X_sm).fit()
print(ols.summary())

**Lectura crítica del summary:**
- Coeficiente de **Newspaper ≈ -0.001**, p-value altísimo (~0.86). **No es estadísticamente distinto de cero.**
- Sin embargo, si entrenamos un modelo con SOLO Newspaper, su coeficiente sale positivo y significativo. ¿Contradicción?
- Respuesta: **multicolinealidad** entre Radio y Newspaper. Lo confirmaremos con VIF en la sección 5.
- R² = 0.897, R²-adj = 0.896 → 89.7% de la varianza de Sales explicada por las 3 inversiones (con 200 observaciones es excelente).

## 4. Ajuste polinomial — cuando la línea recta no alcanza

Si miras con calma TV vs Sales, verás que se aplana en valores altos (rendimientos decrecientes). Probemos con polinomios de varios grados, validados con CV (no en el train).

In [ ]:
X = df[['TV']].values
y = df['Sales'].values

modelos = {
    'Lineal (d=1)':     make_pipeline(PolynomialFeatures(1, include_bias=False), LinearRegression()),
    'Cuadrático (d=2)': make_pipeline(PolynomialFeatures(2, include_bias=False), LinearRegression()),
    'Cúbico (d=3)':     make_pipeline(PolynomialFeatures(3, include_bias=False), LinearRegression()),
    'Grado 5':          make_pipeline(PolynomialFeatures(5, include_bias=False), LinearRegression()),
    'Grado 10 (¡ojo!)': make_pipeline(PolynomialFeatures(10, include_bias=False), LinearRegression()),
}

resultados = []
for nombre, m in modelos.items():
    m.fit(X, y)
    r2_train = m.score(X, y)
    r2_cv = cross_val_score(m, X, y, cv=5, scoring='r2').mean()
    resultados.append({'modelo': nombre, 'R²_train': r2_train, 'R²_CV5': r2_cv})

tabla = pd.DataFrame(resultados)
print(tabla.round(4))

In [ ]:
# Visualización: cómo cada grado ajusta los datos
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
xs = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
for ax, grado in zip(axes, [1, 3, 10]):
    pipe = make_pipeline(PolynomialFeatures(grado, include_bias=False), LinearRegression()).fit(X, y)
    ax.scatter(X, y, alpha=0.5, s=20)
    ax.plot(xs, pipe.predict(xs), color='crimson', lw=2)
    ax.set_title(f'Grado {grado}  ·  R²train = {pipe.score(X,y):.3f}')
    ax.set_xlabel('TV')
axes[0].set_ylabel('Sales')
plt.suptitle('Ajustes polinomiales sobre TV → Sales', y=1.02)
plt.tight_layout(); plt.show()

**Lo que deberías ver:**
- Grado 1: recta, ligero subajuste en valores extremos.
- Grado 3: curva suave, captura saturación en TV alta — buen balance.
- Grado 10: la curva *baila* alrededor de los puntos. R²_train sube; R²_CV se desploma. **Overfitting puro.**

**Regla de oro:** elige el grado más bajo cuya `R²_CV` deje de mejorar (o que la curva ya no muestre patrón claro en los residuos).

## 5. Multicolinealidad y VIF

Definición operativa: VIFⱼ = 1 / (1 − Rⱼ²), donde Rⱼ² es el R² de regresar la variable *j* contra todas las demás.  

| VIF | Interpretación | Acción |
|-----|----------------|--------|
| ≈ 1 | sin colinealidad | OK |
| 1–5 | moderada | vigilar |
| 5–10 | alta | revisar |
| >10 | severa | eliminar / Ridge / Lasso |

In [ ]:
X = df[['TV', 'Radio', 'Newspaper']]
X_const = add_constant(X)

vif = pd.DataFrame({
    'variable': X_const.columns,
    'VIF': [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
})
print(vif.round(3))

**Sorpresa pedagógica:** los VIF de Advertising son bajos (~1.0–1.15). El problema con Newspaper **no es VIF**, es que aporta poco valor único una vez que Radio está en el modelo (Newspaper y Radio comparten el ~25% de variabilidad).

Forcemos un caso sintético para ver VIFs altos en acción:

In [ ]:
# Caso sintético: x2 es casi una copia de x1 (alta colinealidad inducida)
n = 200
x1 = np.random.normal(50, 10, n)
x2 = x1 + np.random.normal(0, 1, n)   # ≈ copia de x1 con ruido pequeño
x3 = np.random.normal(20, 5, n)       # independiente
y_syn = 3 + 0.5*x1 + 0.5*x2 + 0.2*x3 + np.random.normal(0, 2, n)

df_syn = pd.DataFrame({'x1': x1, 'x2': x2, 'x3': x3, 'y': y_syn})
X_syn = add_constant(df_syn[['x1','x2','x3']])
vif_syn = pd.DataFrame({
    'variable': X_syn.columns,
    'VIF': [variance_inflation_factor(X_syn.values, i) for i in range(X_syn.shape[1])]
})
print('VIFs en dataset sintético con x1 ≈ x2:')
print(vif_syn.round(3))

ols_syn = sm.OLS(y_syn, X_syn).fit()
print('\nCoefs y errores estándar:')
print(pd.DataFrame({
    'coef': ols_syn.params.round(3),
    'std_err': ols_syn.bse.round(3),
    'p_value': ols_syn.pvalues.round(4)
}))

**Mira lo que pasó:** los VIF de x1 y x2 explotan a >50. Sus errores estándar son enormes y los coeficientes individuales son inestables, ¡aun cuando R² del modelo entero es altísimo! Eso es lo que el VIF detecta.

**Soluciones:**
1. Eliminar una de las dos variables (la menos relevante para negocio).
2. Combinarlas (`x_combinada = (x1 + x2) / 2`).
3. Usar **Ridge** o **Lasso** (regularización), que reducen la varianza de los coeficientes.

## 6. Diagnóstico de residuales — ¿se cumplen los supuestos del OLS?

Los 4 supuestos: **L**inealidad, **I**ndependencia, **N**ormalidad, **E**quivarianza (homocedasticidad).

In [ ]:
X = df[['TV','Radio','Newspaper']]
X_sm = add_constant(X)
ols = sm.OLS(y, X_sm).fit()

y_hat = ols.fittedvalues
residuos = ols.resid

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# Residuos vs ajustados — verifica linealidad y homocedasticidad
ax[0].scatter(y_hat, residuos, alpha=0.6)
ax[0].axhline(0, color='crimson', linestyle='--')
ax[0].set_xlabel('Valores ajustados (ŷ)')
ax[0].set_ylabel('Residuos')
ax[0].set_title('Residuos vs Ajustados')

# Q-Q plot — verifica normalidad de residuos
sm.qqplot(residuos, line='s', ax=ax[1])
ax[1].set_title('Q-Q plot de residuos')

# Histograma de residuos
ax[2].hist(residuos, bins=25, edgecolor='white')
ax[2].set_xlabel('Residuo')
ax[2].set_title('Distribución de residuos')

plt.tight_layout(); plt.show()

**Lectura del diagnóstico:**
- En el primer panel, los residuos forman una **U invertida** — claramente hay no-linealidad sin capturar (la saturación de TV). Esto justifica probar un término polinomial o de interacción TV×Radio.
- En el Q-Q plot las colas se separan un poco — no es normal estricta, pero sí cercana.
- La dispersión vertical en el primer panel parece relativamente constante, así que la homocedasticidad es razonable.

👉 **Próximo paso natural:** un modelo con `TV + TV² + Radio + TV·Radio`. Lo dejamos para el entregable.

## 7. Entregable parcial S3 (avance)

Sobre **el dataset de tu proyecto**:

1. **Regresión simple**: ajusta una recta sobre 1 predictor numérico relevante de tu objetivo continua. Reporta β₀, β₁, R² y dibuja el scatter con la recta.
2. **Regresión múltiple**: añade ≥ 3 predictores. Reporta el `summary` de `statsmodels.OLS`.
3. **VIF**: calcula VIF de los predictores numéricos. Comenta qué harías si alguno > 10.
4. **Polinomial** *(si aplica)*: si algún scatter individual sugiere curvatura, ajusta polinómica de grado 2 o 3 y compara R²_CV con la lineal.
5. **Interpretación de negocio**: 4–5 líneas explicando qué significan los coeficientes en el contexto de tu pregunta.

Sube el avance al repo del equipo en la rama `s3-modelado`. La entrega completa de la Semana 3 (con clasificación + métricas) cae el viernes 8.

---

## 8. Recordatorio de supuestos OLS — la chuleta

| Supuesto | Cómo se verifica | Si falla… |
|----------|------------------|-----------|
| **Linealidad** | Residuos vs ŷ — sin patrón | Probar polinomial / log / interacción |
| **Independencia** | Durbin-Watson ~ 2; gráfico de residuos en orden | Series temporales → ARIMA / mixed |
| **Normalidad** | Q-Q plot, Shapiro-Wilk | Bootstraps; intervalos de confianza menos fiables |
| **Homocedasticidad** | Residuos vs ŷ no en embudo; Breusch-Pagan | Errores robustos (HC3), o transformar Y (log) |
| **Sin multicolinealidad severa** | VIF < 10 | Eliminar, combinar, o Ridge/Lasso |

Mañana saltamos a **regresión logística** (el primer modelo de clasificación) y aprendemos a balancear clases con **SMOTE**.